In [1]:
import pandas as pd
from pathlib import Path

In [2]:
csv_path = Path("data/id_list.csv")
if not csv_path.is_file():
    csv_path = Path("../data/id_list.csv")

candidates = pd.read_csv(
    csv_path,
    dtype="string",
    skip_blank_lines=False,
    keep_default_na=False,
)

In [3]:
print(candidates.head())
print("Columns:", candidates.columns.tolist())
print("Data rows:", len(candidates))

  tmdb_id
0     155
1     165
2     242
3   49026
4    8077
Columns: ['tmdb_id']
Data rows: 127


Keep the original text and zero-based row index; strip whitespace in a separate working column. CSV record numbers include the header as record 1.

In [4]:
ids = candidates[["tmdb_id"]].copy()
ids = ids.rename(columns={"tmdb_id": "original_id"})
ids.insert(0, "csv_record", ids.index + 2)
ids["stripped_id"] = ids["original_id"].str.strip()


Count empty and whitespace-only IDs.

In [5]:
blank_mask = ids["stripped_id"].eq("")
blank_count = int(blank_mask.sum())
print("Blank rows:", blank_count)


Blank rows: 1


Valid IDs contain only ASCII digits and must be positive. All-zero values are invalid; leading zeros are allowed.

In [6]:
digits_only = ids["stripped_id"].str.fullmatch(r"[0-9]+", na=False)
positive = ids["stripped_id"].str.lstrip("0").ne("")
valid_mask = ~blank_mask & digits_only & positive
malformed_mask = ~blank_mask & ~valid_mask
malformed_count = int(malformed_mask.sum())
print("Malformed nonblank rows:", malformed_count)


Malformed nonblank rows: 3


Normalize valid IDs by removing leading zeros. Count each ID once and count any later occurrences as extra duplicate rows.

In [7]:
ids["normalized_id"] = ids["stripped_id"].where(valid_mask).str.lstrip("0")
valid_ids = ids.loc[valid_mask].copy()
distinct_valid_count = int(valid_ids["normalized_id"].nunique())
extra_duplicate_count = int(valid_ids["normalized_id"].duplicated().sum())
duplicate_rows = valid_ids.loc[
    valid_ids["normalized_id"].duplicated(keep=False)
].sort_values(["normalized_id", "csv_record"])
print("Distinct valid IDs:", distinct_valid_count)
print("Extra duplicate rows:", extra_duplicate_count)


Distinct valid IDs: 117
Extra duplicate rows: 6


Inspect all occurrences of repeated valid IDs, alongside their original text and CSV record positions.

In [8]:
duplicate_rows


,csv_record,original_id,stripped_id,normalized_id
0,2,155,155,155
89,91,155,155,155
53,55,1891,1891,1891
88,90,1891,1891,1891
25,27,278,278,278
58,60,278,278,278
78,80,550,550,550
92,94,550,550,550
38,40,603,603,603
41,43,603,603,603


Reconcile every CSV data row into exactly one category. The total is calculated from the loaded CSV.

In [9]:
observed_count = len(candidates)
reconciled_count = (
    blank_count + malformed_count + extra_duplicate_count + distinct_valid_count
)
print("Observed data rows:", observed_count)
print("Blank rows:", blank_count)
print("Malformed rows:", malformed_count)
print("Extra duplicate rows:", extra_duplicate_count)
print("Distinct valid IDs:", distinct_valid_count)
print(f"Reconciliation: {observed_count} = {blank_count} + {malformed_count} + {extra_duplicate_count} + {distinct_valid_count}")
assert observed_count == reconciled_count, "Row counts do not reconcile"
print("Reconciles: yes")


Observed data rows: 127
Blank rows: 1
Malformed rows: 3
Extra duplicate rows: 6
Distinct valid IDs: 117
Reconciliation: 127 = 1 + 3 + 6 + 117
Reconciles: yes
